# TN1 — GRU ở ngân sách nhỏ

## Câu hỏi

TN1 đến giờ cho một kết quả nhất quán: **hai biến thể LSTM đứng trên cả bốn
biến thể tích chập**, kể cả ModernTCN của năm 2024.

| cấu hình | tham số | cv_mean ± seed_std |
|---|---|---|
| LSTM-352 | 1.502.713 | 0,7570 ± 0,0041 |
| LSTM-67 | 56.908 | 0,7532 ± 0,0020 |
| CNN-LSTM-58 | 55.667 | 0,7527 ± 0,0037 |
| TCN-64 WeightNorm | 150.745 | 0,7463 ± 0,0030 |
| TCN-64 BatchNorm | 151.513 | 0,7423 ± 0,0044 |
| DS-TCN-64 | 56.281 | 0,7421 ± 0,0007 |
| ModernTCN-32 | 56.985 | 0,7405 ± 0,0045 |
| BiLSTM-41 | 57.507 | 0,7398 ± 0,0046 |

Nhưng "LSTM thắng" chưa phải một kết luận sắc. Thắng vì **cơ chế hồi quy có
cổng**, hay vì riêng **cách LSTM cài đặt cơ chế đó**?

GRU tách được hai khả năng ấy, vì nó có cổng nhưng cài khác:

    LSTM   3 cổng (quên, vào, ra) + một ô nhớ riêng tách khỏi trạng thái ẩn
    GRU    2 cổng (đặt lại, cập nhật), KHÔNG có ô nhớ riêng

Nếu GRU cũng lên tới mức LSTM thì thứ có ích là **cơ chế cổng**, và kết luận
của TN1 mạnh hơn hẳn: không phải "LSTM tốt" mà "hồi quy có cổng tốt".

Nếu GRU tụt xuống mức tích chập thì chi tiết cài đặt mới là thứ quyết định, và
phải nói dè dặt hơn nhiều.

## Vì sao 77 chứ không phải 67

Bỏ ô nhớ nên mỗi đơn vị GRU chỉ tốn 3 khối trọng số thay vì 4. Cùng ngân sách
tham số thì GRU chứa được nhiều đơn vị hơn.

    LSTM-67   56.908 tham số
    GRU-77    56.466 tham số     lệch -0,78%

Lệch dưới 1% nên nếu có chênh điểm thì không phải do bên nào được nhiều tham số
hơn. GRU-76 lệch -3,3% và GRU-78 lệch +1,7%, đều xa hơn.

## Đổi đúng một thứ

Mọi thứ khác giữ y nguyên như LSTM-67: 2 tầng, lấy `output[:, -1, :]`,
`Linear(hidden, 25)`, 20 epoch, Adam lr 1e-4, batch 64, MSE, `corr` 0,9, bốn
fold cũ. Chỉ đổi loại tế bào hồi quy.

## 1. Chuẩn bị Colab

Mount Drive để lấy cửa sổ train đã cắt ở `DATA_PREPARE.ipynb`.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

Tải mã nguồn rồi vào thư mục đó. Xem dòng `commit đồ án` để chắc đang chạy bản mới.

In [ ]:
# Xoá trước để chạy lại ô này luôn lấy mã mới nhất, không dính bản cũ.
!rm -rf /content/UWB_RADAR
!git clone -q https://github.com/quangminhho004-blip/UWB_RADAR.git /content/UWB_RADAR
%cd /content/UWB_RADAR
!python scripts/setup_colab.py

Lấy `by_user/` và `windows/` từ Drive. Không cần CSV thô 13 GB.

In [ ]:
!python scripts/restore_processed_data_on_drive.py

## 2. Kiểm bản cài đặt

Bảy phép kiểm, vài giây. Chỗ đáng kiểm nhất ở đây là **lấy đặc trưng**.

Với BiLSTM, `output[:, -1, :]` là SAI vì nửa chiều ngược mới đọc đúng một mẫu.
Với GRU thì `output[:, -1, :]` lại ĐÚNG, vì nó một chiều — bước cuối của chuỗi
cũng là bước cuối cùng nó xử lý.

Hai lớp gần giống nhau mà quy tắc ngược nhau, nên phép kiểm xác nhận thẳng:
`Linear` nhận đúng `hidden` chiều chứ không gấp đôi như BiLSTM, và đầu ra khớp
với `linear(output[:, -1, :])`.

In [ ]:
!python scripts/check_model.py --model gru --hidden 77 \
    --compare-with lstm --compare-hidden 67

## 3. GRU-77 — 4 fold CV, 3 seed

Bảy phép kiểm đạt thì mới chạy.

Tên cấu hình là `gru_h77_mse_corr0.9_seed<N>`. Khác lstm và bilstm, hậu tố
`_h77` LUÔN được ghi: GRU không có cấu hình gốc nào của MobiVital để lấy làm
mặc định, nên "chỉ ghi khi khác mặc định" là vô nghĩa.

Sau mỗi fold script tự nén rồi chép sang Drive. Ngắt phiên giữa chừng thì chạy
lại ô này, fold đã xong được bỏ qua.

12 lần train, khoảng **2 giờ**, ngang LSTM-67.

In [ ]:
!python scripts/run_cv.py --experiment tn1 --model gru --hidden 77 --seed 0
!python scripts/run_cv.py --experiment tn1 --model gru --hidden 77 --seed 1
!python scripts/run_cv.py --experiment tn1 --model gru --hidden 77 --seed 2

## 4. Cất kết quả

Nén lại một lần sau khi xong cả 12 lần chạy, ra tên riêng `tn1_gru_h77.zip`.

In [ ]:
!python scripts/save_results.py tn1 --out tn1_gru_h77

## 5. Bảng so trong phiên này

Ô riêng, không gộp vào ô train: muốn xem lại bảng thì chạy vài giây, chứ không
phải train lại 12 fold.

Chỉ thấy cấu hình của phiên này vì `runs/` vừa bị ô clone xoá. Bảng đủ dựng ở
`TN1_final_evaluation.ipynb`.

In [ ]:
!python scripts/compare_cv.py --experiment tn1

## 6. Ngắt phiên

Colab giữ runtime sau khi ô cuối chạy xong và vẫn tính giờ. Ô này đóng phiên
lại. Kết quả đã nén sang Drive nên ngắt ở đây không mất gì.

Bấm liên tiếp các ô trên thì Colab xếp hàng chạy lần lượt, không cần ngồi canh.

In [ ]:
from google.colab import runtime
runtime.unassign()